# Setup LangSmith API
Retrievals can be traced here for easier debugging.

In [3]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))
import ollama
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGSMITH_API_KEY'] = '123-123-213-123-123-123'

![LangGraph Flow](../../langgraph%20designs/graph_design_v1.png)

# Graph States

In [ ]:
from typing_extensions import TypedDict
from typing import Optional, List, Dict, Any

class GraphState(TypedDict):
    # Core user input
    text_query: str
    image_path: Optional[str]  # Path to uploaded image, if any

    # Routing/intent
    query_type : str # currently being divide into 'emergency'/'Q&A'/'irrelevant'.
    
    # Q&A path
    refined_query: Optional[str]
    queries_for_retrieval: Optional[List[str]]
    current_sub_query: Optional[str]
    retrieved_docs: Optional[List[Dict[str, Any]]]  # Results from retrieval
    reranked_docs: Optional[List[Dict[str, Any]]]   # After rerank step

    # Feedback loop
    retriever: Optional[Any]
    followup_questions: Optional[List[str]]
    user_responses: Optional[List[str]]
    loop_count: int
    hypotheses: Optional[List[str]]  # Current working hypotheses
    next_action: Optional[str]       # What the agent plans to do next ("ask_user", "retrieve", "final_answer", etc.)
    pending_question: Optional[str]  # If the agent wants to ask the user something
    pending_action: Optional[str]    # If the agent wants to perform a tool/action
    user_actions: Optional[List[str]] # Actions the user has taken (e.g., "smelled ear", "provided photo")
    intermediate_thoughts: Optional[List[str]] # Chain-of-thought or reasoning steps

    # Answer generation
    generated_answer: Optional[str]
    hallucination_check: Optional[bool]
    answer_sufficient: Optional[bool]

    # Emergency path
    emergency_instructions: Optional[str]
    emergency_retrieved_docs: Optional[List[Dict[str, Any]]]

    # Web search
    web_search_results: Optional[List[Dict[str, Any]]]

    # Final output
    final_answer: Optional[str]

    # Misc/trace/debug
    path_taken: Optional[List[str]]
    error: Optional[str]

<h1> Graph Nodes

## Query Handler Node 

Before LLM analyze user query and image, it will be assessed with "Is this veterinary-related?". This will ensure our AI tool will not be used for other purpose.

In [ ]:
def query_handler(state):
    text_query = state.get("text_query", "")
    image_path = state.get("image_path", None)

    prompt = (
        "You are a domain classifier for a veterinary assistant. "
        "If an image is provided, understand the image from veterinary point of view."
        "A user query is the combination of text query and image(if there is). "
        "Then, classify the user query into one of three categories:\n"
        "1. 'emergency' — If the user query is about a veterinary emergency (e.g., mass bleeding, serious bone fracture, unconsciousness, severe breathing difficulty, or other life-threatening situations).\n"
        "2. 'Q&A' — If the user query is about is about general veterinary questions, symptom checks, or non-emergency animal health issues.\n\n"
        "3. 'irrelevant' — If the user query is NOT about veterinary, animal health, pet care, etc.\n"
        "Your response must be exactly one of: 'irrelevant', 'emergency', or 'Q&A'. Do not explain your answer or add anything else.\n\n"
        f"User input: {text_query}\n"
    )

    messages = [{
        "role": "user",
        "content": prompt,
        "images": []
    }]

    if image_path and os.path.exists(image_path):
        messages[0]["images"].append(image_path)

    response = ollama.chat(
        model="minicpm-v:8b",
        messages=messages,
        options={"temperature": 0.2}
    )
    result = response['message']['content'].strip().lower()
    # Only allow the three valid outputs
    if result not in ['irrelevant', 'emergency', 'q&a']:
        result = 'irrelevant'
    
    return {"query_type": result}

### test

In [ ]:
def test_query_handler_node(query_handler, test_query, image_path=None):
    # Build the initial state
    state = {
        "text_query": test_query,
        "image_path": image_path
    }
    # Call the query handler node
    new_state = query_handler(state)
    # Print the results
    print("Input Query:", test_query)
    if image_path:
        print("Image Path:", image_path)
    print("Updated State:", new_state)
    print("Query Type:", new_state.get("query_type", "N/A"))
    print("-" * 40)

# --- Example usage ---
test_query_handler_node(query_handler, "What vaccines does my cat need?")
test_query_handler_node(query_handler, "My cat is bleeding a lot after being hit by a car.")
test_query_handler_node(query_handler, "How do I fix my car engine?")
test_query_handler_node(query_handler, "What should I do?", image_path="../emergency_cat.jpg")
test_query_handler_node(query_handler, "What should I feed to this cat?", image_path="../skinny_cat.jpg")


# Q&A Path

## Query Refinement

In [4]:
def get_image_summary(image_path):
    prompt = """From a feline veterinary stand point, provide a highly detailed and objective 
                description of the image. Focus on all observable elements, actions, 
                objects, subjects, their attributes (e.g., color, size, texture), 
                their spatial relationships, and any discernible context or implied scene. 
                Also focus on all possible health issue.
                Describe any text present in the image. This description must be exhaustive 
                and purely factual, capturing every significant visual detail to serve as a 
                comprehensive textual representation for further analysis by another AI model. 
                If the image is entirely irrelevant or contains no discernible subject, 
                state "No relevant visual information."""
    messages = [{
        "role": "user",
        "content": prompt,
        "images": [image_path]
    }]
    response = ollama.chat(
        model="minicpm-v:8b",
        messages=messages,
        options={"temperature": 0.2}
    )
    return response['message']['content']

def query_refinement_node(state):
    text_query = state.get("text_query", "")
    image_path = state.get("image_path", None)
    image_summary = get_image_summary(image_path) if image_path else ""

    # print("******DEBUG QUERY: ", text_query)

    if image_summary:
        prompt = (
        "You are a veterinary assistant AI. Your task is to rewrite and expand the user's question about their cat to make it more effective for searching a veterinary knowledge base.\n\n"
        "You are NOT being asked to give medical advice, make a diagnosis, or recommend treatments.\n\n"
        "Use the image description only to clarify the concern, but **do not invent or assume** any details (such as environment, causes, or severity) not explicitly mentioned by the user or image.\n\n"
        "The refined query must:\n"
        "- Accurately represent the user's concern about their cat\n"
        "- Factually describe any symptoms or visible signs\n"
        "- Include open-ended questions about **possible causes**, **diagnostic steps**, and **general management or prevention**\n"
        "- Remain **neutral and open-ended**, avoiding assumptions or conclusions, stay grounded to user query.\n"
        "- Be phrased as **a single paragraph**, clear and concise, suitable for search retrieval\n"
        "- Do not add new symptoms, behaviors, or environmental details unless they appear in the user query or image description.\n"
        "- Output **only** the refined query — no introductions or explanations\n\n"
        "Here are examples:\n"
        "---\n"
        "User query: My cat keeps shaking its head a lot.\n"
        "Image description: Redness and dark wax visible in one ear.\n"
        "Refined query: My cat has been shaking its head frequently, and I've noticed redness and dark wax in one ear. I'd like to understand what might be causing these symptoms, what diagnostic steps are typically used to evaluate ear conditions in cats, and what general management or preventive options may apply.\n"
        "---\n"
        "User query: My cat has been throwing up for two days.\n"
        "Image description: Pile of partially digested food on carpet.\n"
        "Refined query: My cat has been vomiting for the past two days, with piles of partially digested food. I want to explore potential causes of vomiting in cats, how to tell if it's serious, what diagnostic approaches are used, and general advice for managing this before seeing a vet.\n"
        "---\n"
        f"User query: {text_query}\n"
        f"Image description: {image_summary}\n"
        "Refined query:"
    )
    else:
        prompt = (
        "You are a veterinary assistant AI. Your task is to rewrite and expand the user's question about their cat to make it more effective for searching a veterinary knowledge base.\n\n"
        "You are NOT being asked to give medical advice, make a diagnosis, or recommend treatments.\n\n"
        "The refined query must:\n"
        "- Clearly describe the user's concern about their cat\n"
        "- Remain **neutral and open-ended**, avoiding assumptions or conclusions, stay grounded to user query.\n"
        "- Include helpful questions about **possible causes**, **diagnostic considerations**, and **general management or prevention**\n"
        "- Be phrased as **a single paragraph**, clear and concise, with no extra fluff\n"
        "Do not add new symptoms, behaviors, or environmental details unless they appear in the user query or image description."
        "- Output **only** the refined query — no introductions or explanations\n\n"
        "Here are examples:\n"
        "---\n"
        "User query: I think my cat has a fever, its nose is hot.\n"
        "Refined query: I'm concerned my cat may have a fever because its nose feels hotter than usual. I want to understand what can cause fever in cats, what signs to look for, and what general steps I should take before consulting a veterinarian.\n"
        "---\n"
        "User query: My cat's been sleeping more than usual and not eating.\n"
        "Refined query: My cat is sleeping much more than usual and has lost interest in eating. I'd like to know what potential causes could lead to these symptoms, how to assess if it's urgent, and what general steps I can take before visiting a vet.\n"
        "---\n"
        f"User query: {text_query}\n"
        "Refined query:"
    )

    # print("********DEBUG PROMPT:\n", prompt)
    messages = [{
        "role": "user",
        "content": prompt
    }]
    response = ollama.chat(
        model="mistral:instruct", 
        messages=messages,
        options={"temperature": 0}
    )
    return {"refined_query": response['message']['content']}

### test

In [5]:
def test_query_refinement_node(query_refinement_node, test_query, image_path=None):
    global test_refined_query
    # Build the initial state
    state = {
        "text_query": test_query,
        "image_path": image_path
    }
    # print("*******DEBUG TEST STATE: ",state)

    # Call the query refinement node
    new_state = query_refinement_node(state)

    test_refined_query = new_state.get("refined_query", "N/A")
    # Print the results
    print("Input Query:", test_query)
    if image_path:
        print("Image Path:", image_path)
    print("Refined Query:", new_state.get("refined_query", "N/A"))
    print("-" * 40)

# --- Ears Chapter Example ---
#test_query_refinement_node(query_refinement_node, "What happened to my cat ear? It's being it for a long time. Sometimes I even see blood and wounds in its ear. ", image_path="../cat_ear_problem.jpeg")
test_query_refinement_node(query_refinement_node, "My cat has being scratching its ear too often. There are some dark greasy thing in it. It sratch its ear so often and so hard that I see wounds and blood in it. What should I do?")

# --- Emergency and Infecious Disease Chapter Example ---
# test_query = "My AC is broken, the house is very hot now. I'm worried about my cat in this hot environment. What should I look out for?"
# test_query_refinement_node(query_refinement_node, test_query)

Input Query: My cat has being scratching its ear too often. There are some dark greasy thing in it. It sratch its ear so often and so hard that I see wounds and blood in it. What should I do?
Refined Query:  I'm worried about my cat, as it has been excessively scratching its ear, causing visible wounds and bleeding. There appears to be a dark, greasy substance within the ear canal. I want to understand what could be causing this persistent scratching, how to properly assess the severity of the issue, and what general steps I should take before consulting a veterinarian for treatment options.
----------------------------------------


## Query Decomposition

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def query_decomposition(state):
    refined_query = state['refined_query']

    query_decomposition_prompt = ChatPromptTemplate.from_template(
    """You are a veterinary knowledge assistant. Your task is to break down the following refined query into a list of **concise, semantically rich phrases**, each representing a **different aspect** of the query.

    Guidelines:
    - Each phrase should represent a **distinct, relevant concept** or question implied in the user's concern.
    - Focus on **causes, symptoms, diagnostics, management steps, risk factors, and context**.
    - Avoid repeating or overlapping ideas.
    - Do NOT include general fluff or irrelevant topics.
    - Use phrases or noun-like expressions (not full sentences).
    - Aim for maximum relevance to the original refined query — each phrase should help retrieve part of a comprehensive answer.
    - At the end, include 2-3 **visually grounded search phrases** that require images, diagrams, or visual aids to understand.

    Output only a **JSON array of strings**, with no extra text or explanation.

    Refined query: {refined_query}
    """
    )

    # Create the query decomposition chain
    query_decomposition_chain = (
        query_decomposition_prompt  
        | ChatOllama(model="mistral:instruct")  
        | JsonOutputParser() 
    )

    # --- Demonstration of query decomposition ---

    print(f"Original refined query: {refined_query[:300]} ....")

    decomposed_queries = query_decomposition_chain.invoke({"refined_query": refined_query})
    # Try to extract the JSON array from the response

    print("-" * 80)
    # print(f"Decomposed queries:\n{decomposed_queries}")

    print(f"There are {len(decomposed_queries)} queries after decomposition \n")

    return {"queries_for_retrieval": decomposed_queries}

### Test

In [7]:
def test_query_decomposition(query_decomposition_func, refined_query):
    global test_decomposed_queries
    # Build the initial state
    state = {
        "refined_query": refined_query
    }
    # Call the query decomposition function
    # print("*******DEBUG TEST STATE: ",state)
    new_state = query_decomposition_func(state)
    test_decomposed_queries = new_state['queries_for_retrieval']
    # Print the results
    print("Decomposed Sub-Queries:")
    print(new_state['queries_for_retrieval'])

# --- Example usage ---
test_query_decomposition( query_decomposition, test_refined_query)

Original refined query:  I'm worried about my cat, as it has been excessively scratching its ear, causing visible wounds and bleeding. There appears to be a dark, greasy substance within the ear canal. I want to understand what could be causing this persistent scratching, how to properly assess the severity of the issue, a ....
--------------------------------------------------------------------------------
There are 12 queries after decomposition 

Decomposed Sub-Queries:
['Cat excessive ear scratching', 'Causes of cat ear scratching', 'Cat ear wound from scratching', 'Dark greasy substance in cat ear canal', 'Signs of severe cat ear infection', 'Assessing severity of cat ear issue', 'Pre-veterinary visit steps for cat ear problem', 'Treatment options for cat ear scratching and wounds', 'Risk factors for cat ear infections', 'Visual: Cat ear scratching', 'Visual: Cat ear infection', 'Visual: Normal vs infected cat ear canal']


## Contextual Retrievals

Based on decomposed sub queries, we are able to retrieve contexutally close aligned Documents from the vector database. 

### Setup Unified Retriever (Retrieve text, table, images)

In [8]:
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain_chroma import Chroma
from unified_retriever import UnifiedRetriever
def init_retriever():

    persist_directory = '../../chroma/Ears'
    id_key = "doc_id"

    open_clip_embeddings = OpenCLIPEmbeddings(model_name="ViT-g-14", checkpoint="laion2b_s34b_b88k")

    # Vectorstore for summaries (for similarity search)
    vectorstore = Chroma(
        collection_name="summaries_and_images",
        persist_directory=persist_directory,
        embedding_function=open_clip_embeddings
    )
    # Persistent docstore for originals (all modalities)
    docstore = Chroma(
        collection_name="originals",
        persist_directory=persist_directory,
        embedding_function=open_clip_embeddings
    )

    retriever = UnifiedRetriever(vectorstore, docstore, id_key=id_key)
    return retriever

### Retrieval

In [9]:
seen_doc_ids = set()
all_results = []
retriever = init_retriever()

def contextual_retrieval_flat(state):
    seen_doc_ids = set()
    unique_docs = []
    retriever = init_retriever() 

    for query in state['queries_for_retrieval']:
        results = retriever.retrieve_multi_modal(query, k=5, )
        for res in results:
            doc_id = res.get('doc_id') or res.get('summary_metadata', {}).get('doc_id')
            if doc_id and doc_id not in seen_doc_ids:
                seen_doc_ids.add(doc_id)
                unique_docs.append(res)
    print(f"Total unique documents retrieved: {len(unique_docs)}")

    # No need to append new value to retrieved_docs, LangGraph's state reducer will handle it.
    return {"retriver": retriever, "retrieved_docs": unique_docs}

### test

In [10]:
import copy

def test_contextual_retrieval(queries_for_retrieval):
    global test_retrived_doc

    test_state = {
        "queries_for_retrieval": queries_for_retrieval
    }

    # Use the flat contextual retrieval function
    new_state = contextual_retrieval_flat(test_state)
    unique_docs = new_state["retrieved_docs"]
    print("\nSample of unique retrieved docs:")
    for i, doc in enumerate(unique_docs):
        doc_id = doc.get('doc_id') or doc.get('summary_metadata', {}).get('doc_id')
        print(f"Doc {i}:")
        print(f"  Doc ID: {doc_id}")
        # Check if this is an image context doc
        if doc_id and doc_id.endswith('_context'):
            image_path = doc.get('summary_metadata', {}).get('image_path')
            print(f"  [IMAGE CONTEXT] Points to image file: {image_path}")
        print(f"  Type: {(doc.get('original_metadata') or {}).get('type')}")
        print(f"  Score: {doc.get('score')}")
        # print(f"  Summary: {doc.get('summary')[:100]}...")
        print(f" Original: {retriever.docstore._collection.get(ids=[doc_id], include=["documents"])}")
        print("-" * 40)
    print(f"Total unique docs retrieved: {len(unique_docs)}")
    
    test_retrived_doc = copy.deepcopy(unique_docs)

# Example usage:
test_contextual_retrieval(test_decomposed_queries)

Total unique documents retrieved: 20

Sample of unique retrieved docs:
Doc 0:
  Doc ID: 1d07cec1-76c1-4adc-b22e-e94797ed3b82
  Type: image
  Score: 1.157762885093689
 Original: {'ids': ['1d07cec1-76c1-4adc-b22e-e94797ed3b82'], 'embeddings': None, 'documents': ['./figures/Ears/figure-3-4.jpg'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
----------------------------------------
Doc 1:
  Doc ID: 293bd771-bef8-4ed2-9869-d0f601656337_context
  [IMAGE CONTEXT] Points to image file: None
  Type: image_summary
  Score: 1.1399195194244385
 Original: {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
----------------------------------------
Doc 2:
  Doc ID: 6020df3c-d08f-44e0-a9d6-8dc61b97f677_context
  [IMAGE CONTEXT] Points to image file: None
  Type: image_summary
  Score: 1.1369678974151611
 Original: {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['documents'], 

## ReRank

Retrievals returns docs with high similarities based on cosine-similarity. However, we do need to re-rank their improtance on contexual level.

### Getting image, image_summary pair

In [ ]:
# truly multimodel [monoqwen], use here If running on Nvidia GPU Machine
# pip install "rerankers[monovlm]" qwen-vl-utils transformers
from rerankers import MonoQwen2VLReranker

def rerank_node_monoqwen(state):
    query = state['refined_query']
    candidates = state['retrieved_docs']

    # Prepare candidates for reranker
    rerank_inputs = []
    for doc in candidates:
        if doc.get("modality") == "text":
            rerank_inputs.append(doc["summary"])
        elif doc.get("modality") in ("image", "image_summary"):
            # Use image path if available, else fallback to summary
            image_path = doc.get("original_metadata", {}).get("image_path")
            if image_path:
                rerank_inputs.append(image_path)
            else:
                rerank_inputs.append(doc["summary"])
        else:
            rerank_inputs.append(doc["summary"])

    # Rerank
    from rerankers import MonoQwen2VLReranker
    reranker = MonoQwen2VLReranker.from_pretrained("Qwen/MonoQwen2-VL-v0.1")
    results = reranker.rerank(query, rerank_inputs, top_k=len(rerank_inputs))

    # Attach scores and sort
    for (idx, score) in results:
        candidates[idx]['rerank_score'] = float(score)
    reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

In [ ]:
#Jina Reranker m0. GPU/CPU, but extremly slow in CPU
import base64
import os
from transformers import AutoModel


def image_to_base64(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

def rerank_node_jina_vlm(state):
    query = state['refined_query']
    candidates = state['retrieved_docs']

    documents = []
    doc_types = []
    for doc in candidates:
        if doc.get("modality") in ("image", "image_summary"):
            image_path = doc.get("original_metadata", {}).get("image_path")
            if image_path and os.path.exists(image_path):
                documents.append(image_to_base64(image_path))
                doc_types.append("image")
            else:
                documents.append(doc["summary"])
                doc_types.append("text")
        else:
            documents.append(doc["summary"])
            doc_types.append("text")

    pairs = [[query, doc] for doc in documents]

    model = AutoModel.from_pretrained(
        'jinaai/jina-reranker-m0',
        torch_dtype="auto",
        trust_remote_code=True,
    )
    model.to('cpu')
    model.eval()

    # If most docs are images, use doc_type="image", else "text"
    n_images = doc_types.count("image")
    n_texts = doc_types.count("text")
    doc_type = "image" if n_images > n_texts else "text"

    # If mixed, filter and rerank separately, then merge (advanced)
    # For now, just use the dominant type
    scores = model.compute_score(pairs, max_length=2048, doc_type=doc_type)

    for doc, score in zip(candidates, scores):
        doc['rerank_score'] = float(score)
    reranked = sorted(candidates, key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

In [ ]:
def rerank_node_hybrid_v2(state):
    """
    General-purpose reranking using Qwen3-Reranker-0.6B for multi-modal retrieval.
    - Text docs: use summary only.
    - Image docs: only rerank if a matching image_summary exists (doc_id+'_context').
    - Image_summary: use summary directly.
    - All others: skip.
    """
    instruction = (
        "You are a veterinary assistant AI. "
        "Given a user query about animal health, carefully consider what the user is asking and what information they are seeking. "
        "Rank the following passages and image descriptions by how contextually relevant and helpful they are for answering the user's specific question or concern. "
        "Consider both direct answers and supporting information that would help the user understand, diagnose, or manage the situation. "
        "Prioritize passages and images that provide clear, actionable, step-by-step, or practical advice. "
        "If an image description offers a visual guide, demonstration, or helps clarify a procedure or symptom, treat it as highly relevant. "
        "Do not ignore image summaries—visual information can be as important as text for veterinary advice."
    )
    query = state.get('refined_query', '')
    candidates = state.get('retrieved_docs', [])
    # No need for docstore now

    # Build a lookup for image_summary docs by doc_id
    image_summary_lookup = {}
    for doc in candidates:
        modality = doc.get('modality') or (doc.get('original_metadata') or {}).get('type')
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        if modality == 'image_summary' and doc_id:
            image_summary_lookup[doc_id] = doc

    rerank_inputs = []
    doc_indices = []

    for idx, doc in enumerate(candidates):
        modality = doc.get('modality') or (doc.get('original_metadata') or {}).get('type')
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        if not doc_id:
            continue
        if modality == 'text':
            doc_text = doc.get("summary", "")
        elif modality == 'image':
            summary_id = doc_id + '_context'
            summary_doc = image_summary_lookup.get(summary_id)
            if summary_doc and summary_doc.get('summary'):
                doc_text = summary_doc['summary']
            else:
                continue  # skip image if no summary
        elif modality == 'image_summary':
            doc_text = doc.get('summary', '')
        else:
            continue  # skip other modalities
        if not doc_text or not isinstance(doc_text, str):
            continue
        input_str = f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc_text}"
        rerank_inputs.append(input_str)
        doc_indices.append(idx)

    if not rerank_inputs:
        return {"reranked_docs": []}

    # Load Qwen3-Reranker-0.6B (SequenceClassification)
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    import numpy as np

    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-0.6B")
    model = AutoModelForSequenceClassification.from_pretrained("Qwen/Qwen3-Reranker-0.6B")
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Batch inference
    with torch.no_grad():
        batch = tokenizer(rerank_inputs, padding=True, truncation=True, max_length=2048, return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        scores = outputs.logits.squeeze(-1).cpu().numpy()

    for idx, score in zip(doc_indices, scores):
        if isinstance(score, np.ndarray):
            score = float(score.flatten()[0])
        candidates[idx]['rerank_score'] = float(score)
    reranked = sorted([candidates[i] for i in doc_indices], key=lambda x: x.get('rerank_score', 0), reverse=True)
    return {"reranked_docs": reranked}

## test

In [ ]:
def test_rerank_node(
    rerank_node, refined_query, 
    retrieved_docs, retriever=None, top_n=5
):
    """
    Test the rerank_node
    - rerank_node: the rerank function (e.g., rerank_node_hybrid_v2)
    - refined_query: the query string
    - retrieved_docs: list of docs to rerank
    - retriever: retriever instance (should have .docstore)
    - top_n: how many top docs to print
    """
    # Build the state as expected by rerank_node_hybrid_v2
    state = {
        "refined_query": refined_query,
        "retrieved_docs": retrieved_docs,
        "retriever": retriever,
    }
    # Call the rerank node
    new_state = rerank_node(state)
    reranked_docs = new_state.get("reranked_docs", [])
    print(f"Total docs after reranking: {len(reranked_docs)}")
    print(f"Top {top_n} reranked docs (by rerank_score):\n")
    for i, doc in enumerate(reranked_docs[:top_n]):
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        modality = (doc.get('original_metadata') or {}).get('type') or doc.get('modality')
        score = doc.get('rerank_score')
        print(f"Doc {i}:")
        print(f"  Doc ID: {doc_id}")
        print(f"  Type: {modality}")
        print(f"  Rerank Score: {score}")
        summary = doc.get('summary', '')
        # For text docs, try to fetch original text from docstore if available
        if modality == "text" and retriever is not None:
            try:
                original = retriever.docstore._collection.get(ids=[doc_id], include=["documents"])
                original_text = original["documents"][0] if original and original.get("documents") else None
            except Exception:
                original_text = None
            if not original_text:
                original_text = summary
            print("  Original Text:")
            print(f"    {original_text[:300]}{'...' if len(original_text) > 300 else ''}")
        else:
            print("  Summary:")
            print(f"    {summary[:300]}{'...' if len(summary) > 300 else ''}")
        print("-" * 40)
    # Check sorting
    scores = [doc.get('rerank_score') for doc in reranked_docs if doc.get('rerank_score') is not None]
    if scores and scores == sorted(scores, reverse=True):
        print("PASS: Docs are sorted by rerank_score descending.")
    else:
        print("FAIL: Docs are not sorted correctly or scores are missing.")
    return reranked_docs  # Optionally return for further use

# Example usage:
reranked_docs = test_rerank_node(rerank_node_hybrid_v2, test_refined_query, test_retrived_doc, retriever)

# Relevancy_check_node


In [13]:
def relevancy_check_node(state):
    """
    For each doc in reranked_docs, ask an LLM if it is relevant and useful for answering the refined_query.
    Only keep docs the LLM says are relevant.
    """
    relevant_docs = []
    query = state.get('refined_query')
    docs = state.get('retrieved_docs')
    
    for doc in docs:
        # Use summary for text, or summary/image path for images
        modality = doc.get('modality') or (doc.get('original_metadata') or {}).get('type')
        summary = doc.get('summary', '')
        if modality in ('image', 'image_summary'):
            image_path = (doc.get('original_metadata') or {}).get('image_path')
            doc_desc = f"[IMAGE] Path: {image_path}\nSummary: {summary}"
        elif modality == 'table':
            doc_desc = f"[TABLE] Summary: {summary}"
        else:
            doc_desc = f"[TEXT] Summary: {summary}"

        prompt = (
            "You are a veterinary assistant AI. You are checking if a document is relevant and useful for answering a user's veterinary question. "
            "The document may be a summary of a textbook passage, a table, or an image (with a summary). "
            "Only say YES if the document contains information that would help answer the user's question, or provides context, steps, or background. "
            "If the document is off-topic, generic, or not helpful, say NO.\n\n"
            f"User query: {query}\nDocument: {doc_desc}\n\n"
            "Is this document relevant and useful for answering the query?"
            "Respond with only YES or NO."
        )
        messages = [{"role": "user", "content": prompt}]
        response = ollama.chat(
            model="mistral:instruct",
            messages=messages,
            options={"temperature": 0},
        )
        answer = response['message']['content'].strip().lower()
        print("🌟 Answer: ", answer)
        if answer.startswith('yes'):
            relevant_docs.append(doc)
        
    print(f"Relevancy check: {len(relevant_docs)} of {len(docs)} docs kept.")
    state['relevant_docs'] = relevant_docs
    return state

In [14]:
def test_relevancy_check_node():
    global test_relevant_docs
    """
    Test the relevancy_check_node with a mock state containing reranked_docs.
    """
    mock_state = {
        "refined_query": test_refined_query,
        "retrieved_docs": test_retrived_doc,
    }
    print("--- Before relevancy_check_node ---")
    print(f"Docs in: {len(mock_state['retrieved_docs'])}")
    new_state = relevancy_check_node(mock_state)
    print(f"Docs out: {len(new_state['relevant_docs'])}")
    test_relevant_docs = new_state['relevant_docs']

# Example usage:
test_relevancy_check_node()

--- Before relevancy_check_node ---
Docs in: 20
🌟 Answer:  yes
🌟 Answer:  yes
🌟 Answer:  no, the provided document does not contain information that would help answer the user's question about their cat's ear problem. the image shows a bandaged cat head, but it does not provide any context or guidance related to excessive scratching, visible wounds, bleeding, dark greasy substances in the ear canal, or assessing the severity of an ear infection in cats.
🌟 Answer:  yes
🌟 Answer:  no
🌟 Answer:  yes
🌟 Answer:  no, the document does not provide information that would help answer the user's question about their cat's ear problem. the document discusses ear growths in pets, which may be cancerous, but it does not address excessive scratching, visible wounds, bleeding, or dark greasy substances within the ear canal, which are the symptoms described by the user.
🌟 Answer:  yes
🌟 Answer:  yes
🌟 Answer:  yes
🌟 Answer:  no, the provided image does not contain information that would help answer th

# Thinking Node

This step is to take all on-hand info and reranked doc to make analysis. Think about user's intent, what they want to know, what they need to know, also what AI need to know.

In [19]:
import re
def thinking_node(state):
    """
    Given the current state, use Qwen3 to reason step-by-step about how to answer the user's question.
    Updates state with intermediate thoughts, hypotheses, and next_action if more info or tools are needed.
    """
    user_query = state.get("text_query", "")
    image_summary = state.get("image_summary", "") #if user_query contain image
    relevant_docs = state.get("relevant_docs")

    prompt = (
        "You are a veterinary assistant AI. The user is a pet owner with little veterinary knowledge. "
        "Explain in simple, actionable language, only suggesting home-care steps. If the case is serious, remind the user to see a vet. "
        "Base your answer strictly on the provided docs. "
        "If you need more info, specify what and which tool to use. "
        "You have access to a veterinary textbook and a database of documents, images, and tables. "
        "If the provided docs do not fully answer the user's question, you can suggest new search queries to retrieve more information. "
        "To do this, output 'Next action: retrieve more info' and provide a list of new queries that would help you find the answer. "
        "Example queries: [\"causes of cat ear bleeding\", \"cat ear infection symptoms\", \"treatment for dark wax in cat ear\"]\n\n"

        "When to use each action/tool:\n"
        "- Use 'retrieve more info' if the provided docs are insufficient, missing key details, or you are uncertain about the answer. Suggest specific, focused queries that would help you find the missing information in the textbook or database.\n"
        "- Use 'ask the user a question' if you need clarification, more details about the pet's symptoms, or additional context from the user to proceed.\n"
        "- Use 'ready to answer' if you have enough information from the provided docs to give a helpful, actionable answer.\n"
        "If you are missing key details, or the docs are insufficient, do not guess—ask for more info or suggest retrieval.\n\n"

        "Respond in JSON, using one of these formats:\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"retrieve more info\",\n'
        '  \"queries\": [\"query1\", \"query2\"],\n'
        '  \"user_response\": \"your answer\"\n'
        '}\n'
        "or\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"ask the user a question\",\n'
        '  \"user_response\": \"your answer\"\n'
        '}\n'
        "or\n"
        '{\n'
        '  \"thinking\": \"your reasoning\",\n'
        '  \"next_action\": \"ready to answer\",\n'
        '  \"user_response\": \"your answer\",\n'
        '  \"images\": [\"image_path1\", \"image_path2\"]\n'
        '}\n\n'
        f"User question: {user_query}\n"
    )

    #If user query has a photo
    if image_summary: 
        prompt += f"Image summary: {image_summary}\n"

    prompt += "Relevant information from veterinary handbook:\n"
    for i, doc in enumerate(relevant_docs):
        modality = doc.get('modality') or (doc.get('original_metadata') or {}).get('type')
        doc_id = doc.get('doc_id') or (doc.get('original_metadata') or {}).get('doc_id')
        if modality in ('image', 'image_summary'):
            original_metadata = doc.get('original_metadata') or {}
            # Convert to pretty JSON string for LLM readability
            import json
            metadata_str = json.dumps(original_metadata, indent=2, ensure_ascii=False)
            prompt += (
                f"{i+1}. [IMAGE] The following is the original_metadata for this image (summary and image_path are included):\n"
                f"{metadata_str}\n(id: {doc_id})\n"
            )
        elif modality == 'table':
            summary = doc.get('summary', '')
            prompt += f"{i+1}. [TABLE] Summary: {summary} (id: {doc_id})\n"
        else:
            summary = doc.get('summary', '')
            prompt += f"{i+1}. [TEXT] Summary: {summary} (id: {doc_id})\n"

    messages = [{"role": "user", "content": prompt}]
    response = ollama.chat(
        model="qwen3:8b", 
        messages=messages,
        options={"temperature": 0.2},
    )
    llm_output = response['message']['content']

    think_match = re.search(r"<think>(.*?)</think>", llm_output, re.DOTALL | re.IGNORECASE)
    if think_match:
        reasoning = think_match.group(1).strip()
    else:
        reasoning = ""  # fallback if not found

    # Optionally, extract the rest (user-facing answer)
    user_response = re.sub(r"<think>.*?</think>", "", llm_output, flags=re.DOTALL | re.IGNORECASE).strip()

    print("✨Thinking Node✨: ", reasoning, "\n")
    print("✨User Response✨: ", user_response, "\n")

    state["intermediate_thoughts"] = state.get("intermediate_thoughts", []) + [reasoning]
    state["last_user_response"] = user_response  

    # Try to extract next_action from the output (simple heuristic, can be improved)
    if "retrieve more info" in llm_output.lower():
        state["next_action"] = "retrieve_more_info"
    elif "interpret a new image" in llm_output.lower() or "interpret new image" in llm_output.lower():
        state["next_action"] = "interpret_image"
    elif "ask the user" in llm_output.lower() or "ask user" in llm_output.lower():
        state["next_action"] = "ask_user"
    else:
        state["next_action"] = None

    # Optionally, extract hypotheses if the LLM lists them
    if "hypothesis" in llm_output.lower():
        state["hypotheses"] = [llm_output]

    return state

## Tools

In [ ]:
def book_retrieval_for_more(query, retriever, k=5):
    print("🔩Book Retriever Tool Called! 🔧")

def interpret_image(image_path, model = "minicpm-v:8b"):
    print("📷Image Interpreter Tool Called! ⛰️")


In [20]:
def test_thinking_node():
    """
    Test the thinking_node with a mock state including text query, optional image summary, and mixed retrieved_docs.
    Prints the updated state, intermediate thoughts, and next_action.
    """
    # Example mock state
    mock_state = {
        "text_query": test_refined_query,
        # Uncomment the next line to test with an image summary
        # "image_summary": "The image shows a cat's ear with visible dark debris and some redness.",
        "relevant_docs": test_relevant_docs
    }
    print()
    thinking_node(mock_state)


# Example usage:
test_thinking_node()


✨Thinking Node✨:  Okay, let's tackle this user's question. They have a cat that's been scratching its ear a lot, causing wounds and bleeding, with dark, greasy stuff in the ear canal. They want to know possible causes, how to assess severity, and home steps before seeing a vet.

First, I need to check the provided docs. The relevant info includes images and texts. The images mention hematoma from head shaking, which could be related. The text talks about ear mites in younger cats, infections from skin issues, and signs like discharge, scratching, and tenderness. Also, there's info on cleaning ears and avoiding harsh substances.

The user's cat has visible wounds and bleeding, which might indicate an infection or trauma. The dark, greasy substance could be wax, debris, or an infection. The docs mention that severe skin abrasion can lead to infection or abscess, so that's a concern. The hematoma image shows swelling from head shaking, which might be a possibility here.

But the user nee